# Step 2: Gauge Mechanism Routing

## tl;dr

这个 notebook 不预设 LocalGauge 正确。它用同一批严格正交 gauge 和 paired
tiny probes，依次检查 exactness、local/global 与感受野、短训/长训、二阶
统计、time-bin 偏好和 decoder 对真实 probe residual 的放大，并输出下一步
分流表。

默认设置是可运行 smoke。只有 `SEEDS>=3`、更长预算和正式生成实验才能形成
研究结论。

## Context & Methods

### Key assumptions

- `A` 是固定、低容量、严格正交的，不联合训练。
- train 只更新 probe；validation 只比较 gauge。
- local/global 不是严格 matched-FLOPs，因此它是机制路由证据；最终必须加
  matched-FLOPs attention 和 cross-architecture crossover。
- decoder 分支使用 probe 实际预测出的 `z0` residual，不使用各向同性随机噪声。

In [ ]:
from pathlib import Path
import sys
from IPython.display import display

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from experiments.architecture_gauge import (
    CodecDataConfig, GaugeSpec, ProbeConfig, ProbeTrainingConfig,
    add_identity_ratios, decoder_residual_table, exact_equivalence_table,
    finite_difference_headroom, mechanism_routing_table,
    plot_decoder_residuals, plot_learning_curves, plot_locality_comparison,
    plot_second_order_control, plot_time_bin_heatmap,
    prepare_codec_data, probe_configs, reconstruction_equivalence_table,
    run_probe_grid,
)

### Experiment parameters

In [ ]:
MODEL_KEY = "rae_dinov2"
DATASET_NAME = "imagenet_parquet"
DATASET_PATH = "/data/shared/imagenet-1k"
TRAIN_COUNT, VAL_COUNT = 24, 12
DEVICE = "cuda:0"
SEED = 0

# Smoke: 12 steps, one seed. Research run: >=500 steps and SEEDS=(0,1,2).
STEPS = 12
SEEDS = (0,)
HIDDEN = 16
HEADROOM_DELTA = 0.25

GAUGES = [
    GaugeSpec("identity"),
    GaugeSpec("allpass_plus", kind="fourier_allpass", strength=+HEADROOM_DELTA, radius=1),
    GaugeSpec("allpass_minus", kind="fourier_allpass", strength=-HEADROOM_DELTA, radius=1),
    GaugeSpec("allpass_r3", kind="fourier_allpass", strength=0.65, radius=3),
    GaugeSpec("channel_0.5", kind="channel_givens", strength=0.5, seed=SEED),
    GaugeSpec("haar_2x2", kind="block_haar"),
]
PROBES = probe_configs(hidden=HIDDEN)
TRAINING = ProbeTrainingConfig(
    steps=STEPS,
    eval_steps=(0, max(1, STEPS // 3), max(2, 2 * STEPS // 3), STEPS),
    batch_size=min(4, TRAIN_COUNT),
    eval_batches=3,
    time_bins=6,
    seed=SEED,
)

## Data

### 1. Frozen codec and disjoint ImageNet splits

In [ ]:
data = prepare_codec_data(CodecDataConfig(
    dataset_name=DATASET_NAME,
    dataset_path=DATASET_PATH,
    train_split="train",
    val_split="validation",
    train_count=TRAIN_COUNT,
    val_count=VAL_COUNT,
    model_key=MODEL_KEY,
    device=DEVICE,
    seed=SEED,
))
print("train z:", tuple(data.train_latents.shape), "val z:", tuple(data.val_latents.shape))
print("train/val source splits: train / validation")

## Results

### 2. Exact gate and second-order controls

all-pass 若在 probe 上产生差异，但 `total_psd_rel_error` 仍接近零，说明差异
不能由总功率谱解释。它仍不自动证明 higher-order locality 是唯一原因。

In [ ]:
exact = exact_equivalence_table(data.val_latents, GAUGES, device=DEVICE)
recon_exact = reconstruction_equivalence_table(data, GAUGES, count=2)
display(exact.round(8))
display(recon_exact.round(8))
assert exact[["inverse_rel_l2", "norm_rel_error", "paired_noise_rel_error"]].to_numpy().max() < 1e-5

### 3. Paired probe grid

`local_rf5` 与 `local_rf9` 在同一卷积族内改变感受野；`global_attn` 提供全局
读取对照。表中同时显示参数量，避免把容量差异藏起来。

In [ ]:
runs, history, time_rows = run_probe_grid(
    data.train_latents,
    data.val_latents,
    GAUGES,
    PROBES,
    TRAINING,
    seeds=SEEDS,
    device=DEVICE,
    latent_scale=data.latent_scale,
)

probe_sizes = history[["probe", "probe_kind", "receptive_field", "parameter_count"]].drop_duplicates()
final_ratios = add_identity_ratios(history)
final_ratios = final_ratios[final_ratios["step"] == STEPS]
display(probe_sizes)
display(final_ratios[["probe", "gauge", "seed", "relative_mse", "loss_ratio_to_identity"]].round(5))

### 4. Finite-horizon and locality views

左图回答差异是否随训练预算消失；右图回答同一 gauge 对不同感受野/全局
probe 的影响是否系统不同。identity 水平线固定为 1。

In [ ]:
plot_learning_curves(history);
plot_locality_comparison(history);

### 5. Identity-neighborhood directional check

In [ ]:
headroom = finite_difference_headroom(
    history,
    plus_gauge="allpass_plus",
    minus_gauge="allpass_minus",
    delta=HEADROOM_DELTA,
)
display(headroom.round(6))

### 6. Can second-order statistics explain probe sensitivity?

这张散点图是控制图，不做线性因果推断。关键读法是：PSD/Gram 误差接近
数值零时，probe ratio 是否仍明显偏离 1。

In [ ]:
plot_second_order_control(exact, history);

### 7. Noise-level preference

同一 gauge 在低噪声和高噪声 bin 中若稳定出现相反方向，才有理由考虑
time-dependent gauge。单 seed 的局部翻转只算提示。

In [ ]:
plot_time_bin_heatmap(time_rows, probe="local_rf5");
plot_time_bin_heatmap(time_rows, probe="global_attn");

### 8. Decoder amplification from actual probe residuals

只解码 identity 与强 all-pass 的 local/global 结果，避免 notebook 被 decoder
可视化拖慢。横轴是预测 `z0` 的 latent RMSE，纵轴是对应 decoded image L1。

In [ ]:
decoder_rows = decoder_residual_table(
    data,
    runs,
    TRAINING,
    count=min(3, VAL_COUNT),
    run_filter={
        ("local_rf5", "identity"),
        ("local_rf5", "allpass_r3"),
        ("global_attn", "identity"),
        ("global_attn", "allpass_r3"),
    },
)
display(decoder_rows.round(6))
plot_decoder_residuals(decoder_rows);

### 9. Mechanism routing table

路由阈值是项目管理启发式，不是统计检验。`exploratory (<3 paired seeds)`
的任何 supported/candidate 都必须升级为多 seed 长预算后再使用。

In [ ]:
routing = mechanism_routing_table(
    history,
    time_rows,
    exact,
    decoder_rows,
    improvement_threshold=0.02,
    locality_threshold=0.03,
)
display(routing)

## Takeaways

- exact gate 失败：没有机制结论，先修实现。
- 只有 non-identity 变差：支持 H1，不支持 Architecture-Optimal Gauge。
- all-pass 二阶统计不变，且 local penalty 随 RF/global 明显收缩：进入 static
  LocalGauge 与跨架构交换实验。
- 差异随训练消失：只主张 finite-horizon efficiency。
- latent error 接近但 decoded error 分离：将 decoder-aware 作为独立方向。
- time-bin 方向稳定反转：才评估 moving gauge。
- 多方向、多 seed、长预算均无法 beat identity：停止质量方法。

当前 notebook 的默认 smoke 只验证管线和可视化，不自动证明上述任一机制。